# Step 27 — Extraction improvement pilot

**Settings: `GPU T4 x2`, `Internet On`.**
**Inputs:** competition data, `rsna-knee-03-extract-all`

## Why labels now

Your ensemble scores **0.835** on the gold holdout. Your extraction quality is **0.838**. The model has caught up to its own training signal — so from here, better labels lift everything.

The clearest case is Effusion: model **0.746** against extraction **0.793**. It's label-limited, not model-limited. No architecture, resolution or ensemble can fix that, because the information isn't in the target.

## Four configurations, tested on 58 studies before spending real GPU time

| config | generations | est. time |
|---|---|---|
| A — 7B greedy (reproduces your 0.838) | 116 | ~4 min |
| B — 7B self-consistency x3 | 348 | ~13 min |
| C — 14B greedy | 116 | ~8 min |
| D — 14B self-consistency x3 | 348 | ~24 min |

Under an hour total. Only then do we commit to re-extracting all 4,407 — which costs 10 GPU-hours at 14B greedy, or 31 hours with self-consistency.

**Self-consistency** samples each report several times at non-zero temperature and averages. Individual readings vary on ambiguous phrasing; the average is steadier. It changes nothing about the prompt.

**A larger model** should help most on the harder multilingual reports, where 7B is likeliest to misread idiom.

## What we're deliberately not testing yet

**Few-shot prompting** with the gold labels would be the third lever, and the most likely to help with threshold calibration — the LLM currently guesses where annotators drew the line on "mild" osteoarthritis. But putting gold examples in the prompt means they can no longer serve as an unbiased measurement, and those 58 studies are the only expert-labelled data you have. We test the two free options first, and only spend gold if they fall short.

## Threshold to beat

**0.838.** Anything under +0.01 isn't worth 10 GPU-hours plus retraining five folds.

### 1 · Setup

In [ ]:
import os, re, gc, json, time, glob, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score

ROOT   = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train  = pd.read_csv(f"{ROOT}/train.csv")
gold   = pd.read_csv(glob.glob("/kaggle/input/notebooks/*/*/gold58.csv")[0])
LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()

KEYMAP = {"acl":"ACL","mcl":"MCL","medial_meniscus":"Medial Meniscus",
          "lateral_meniscus":"Lateral Meniscus","medial_oa":"Medial OA",
          "lateral_oa":"Lateral OA","pf_oa":"PF OA","effusion":"Effusion",
          "synovitis":"Synovitis","bakers":"Baker's","contusion":"Contusion",
          "fracture":"Fracture"}
KEYS = list(KEYMAP)

gl_ids  = gold["StudyInstanceUID"].tolist()
reports = train.set_index("StudyInstanceUID").loc[gl_ids, "Report"].astype(str).tolist()
Ytrue   = gold.set_index("StudyInstanceUID").loc[gl_ids, LABELS].values.astype(int)
print(f"gold studies: {len(reports)} | GPU: {torch.cuda.get_device_name(0)}")

### 2 · Prompts — unchanged from the production extractor

In [ ]:
SYS_CAT = """You are a musculoskeletal radiologist extracting structured findings from knee MRI reports.
The report may be in ANY language. Read it in its original language.

For each finding, decide: PRESENT, ABSENT (negated/normal/intact), or NOT_MENTIONED.

- acl: anterior cruciate ligament tear or injury
- mcl: medial collateral ligament tear or injury
- medial_meniscus: medial meniscus tear
- lateral_meniscus: lateral meniscus tear
- medial_oa: MEDIAL tibiofemoral osteoarthritis / chondral loss
- lateral_oa: LATERAL tibiofemoral osteoarthritis / chondral loss
- pf_oa: patellofemoral osteoarthritis / chondromalacia
- effusion: joint effusion
- synovitis: synovial thickening or inflammation
- bakers: Baker's / popliteal cyst
- contusion: bone contusion, bruise, marrow oedema from trauma
- fracture: fracture of any bone

RULES:
1. Negation decides. "ACL intact", "menisci normal", "no tear" => absent.
2. Degenerative signal that is NOT a tear => absent for meniscus/ligament.
3. Distinguish MEDIAL from LATERAL carefully.
4. Never guess: if not referred to, say not_mentioned.

Reply ONLY JSON:
{"acl":"present|absent|not_mentioned","mcl":"...","medial_meniscus":"...","lateral_meniscus":"...","medial_oa":"...","lateral_oa":"...","pf_oa":"...","effusion":"...","synovitis":"...","bakers":"...","contusion":"...","fracture":"..."}"""


SYS_NUM = """You are a musculoskeletal radiologist reading knee MRI reports and estimating how likely each of twelve findings is to be truly present.

The report may be in ANY language. Read it in its original language.

For EACH finding give an integer 0-100: the likelihood an expert reviewing the actual images would mark it present.

  0-10    explicitly negated ("ACL intact", "menisci normal", "no effusion")
  11-30   not mentioned at all
  31-55   equivocal, borderline, trace, minimal, degenerative change short of a tear
  56-80   clearly described but mild, or described indirectly
  81-100  explicitly and unambiguously present

THE TWELVE FINDINGS (match on MEANING, not exact wording):
acl              ACL tear, rupture, discontinuity
mcl              MCL tear, rupture, high-grade sprain
medial_meniscus  medial meniscus TEAR (degeneration alone is NOT a tear -> low)
lateral_meniscus lateral meniscus TEAR (same rule)
medial_oa        MEDIAL compartment: osteoarthritis, chondral loss/thinning, joint space
                 narrowing, subchondral sclerosis, osteophytes
lateral_oa       LATERAL compartment: same features
pf_oa            PATELLOFEMORAL (patella/trochlea): same, chondromalacia patellae,
                 retropatellar cartilage thinning
effusion         joint effusion, intra-articular fluid (trace/small counts, ~50-65)
synovitis        synovial thickening, enhancement, inflammation, plica synovitis
bakers           Baker's cyst, popliteal cyst, gastrocnemio-semimembranosus bursa cyst
contusion        bone contusion, bruise, marrow oedema from trauma
fracture         fracture, cortical break, avulsion, insufficiency/stress fracture

RULES:
- Negation is decisive: "intact", "normal", "unremarkable", "no evidence of" => under 10.
- "Tricompartmental osteoarthritis" raises ALL THREE OA findings.
- Meniscal degeneration without a tear line => 20-40.
- If never referred to, give 15-25. Never exactly 0 for silence.

Reply ONLY JSON of twelve integers:
{"acl":0,"mcl":0,"medial_meniscus":0,"lateral_meniscus":0,"medial_oa":0,"lateral_oa":0,"pf_oa":0,"effusion":0,"synovitis":0,"bakers":0,"contusion":0,"fracture":0}"""

CAT2NUM = {"present": 90.0, "not_mentioned": 20.0, "absent": 5.0}
W = {"acl":0.80,"mcl":0.80,"medial_meniscus":0.35,"lateral_meniscus":0.35,
     "medial_oa":0.20,"lateral_oa":0.20,"pf_oa":0.20,"effusion":0.20,
     "synovitis":0.35,"bakers":0.50,"contusion":0.50,"fracture":0.50}
print("prompts ready")

### 3 · Runner

One function for all four configs. `n_samples > 1` switches to sampling at temperature 0.7 and averages the results — that's self-consistency.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
os.system("pip install -q -U bitsandbytes 2>/dev/null")

_loaded = {}
def get_model(name):
    if name in _loaded:
        return _loaded[name]
    for k in list(_loaded):                      # only one model on the GPU at a time
        del _loaded[k]
    gc.collect(); torch.cuda.empty_cache()
    print(f"loading {name} ...", flush=True)
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    m = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb,
                                             device_map={"": 0}, dtype=torch.float16)
    m.eval()
    print(f"  loaded in {time.time()-t0:.0f}s")
    _loaded[name] = (tok, m)
    return tok, m


def generate(model_name, system, reports, n_samples=1, max_new=220, bs=8):
    tok, model = get_model(model_name)
    outs = [[] for _ in reports]
    order = sorted(range(len(reports)), key=lambda i: len(reports[i]))
    for rep in range(n_samples):
        for b0 in range(0, len(order), bs):
            idx = order[b0:b0+bs]
            texts = [tok.apply_chat_template(
                        [{"role":"system","content":system},
                         {"role":"user","content":f"REPORT:\n{reports[i][:6000]}"}],
                        tokenize=False, add_generation_prompt=True) for i in idx]
            enc = tok(texts, return_tensors="pt", padding=True,
                      truncation=True, max_length=3072).to(model.device)
            kw = dict(max_new_tokens=max_new, pad_token_id=tok.pad_token_id)
            if n_samples > 1:
                kw.update(do_sample=True, temperature=0.7, top_p=0.9)
            else:
                kw.update(do_sample=False, temperature=None, top_p=None, top_k=None)
            with torch.no_grad():
                g = model.generate(**enc, **kw)
            new = g[:, enc.input_ids.shape[1]:]
            for j, i in enumerate(idx):
                txt = tok.decode(new[j], skip_special_tokens=True)
                m = re.search(r"\{.*?\}", txt, re.S)
                try:    outs[i].append(json.loads(m.group(0)) if m else {})
                except Exception: outs[i].append({})
    return outs


def cnum(d, k): return CAT2NUM.get(str(d.get(k, "not_mentioned")).lower(), 20.0)
def nnum(d, k):
    try:    return min(max(float(d.get(k, 20)), 0), 100)
    except Exception: return 20.0

def score(cat_runs, num_runs):
    """Average across samples, blend the two prompts, return macro AUC + per-label."""
    per = []
    for ci, k in enumerate(KEYS):
        c = np.mean([[cnum(d, k) for d in runs] for runs in cat_runs], axis=1)
        n = np.mean([[nnum(d, k) for d in runs] for runs in num_runs], axis=1)
        s = W[k] * c + (1 - W[k]) * n
        y = Ytrue[:, ci]
        per.append(np.nan if y.min() == y.max() else roc_auc_score(y, s))
    return float(np.nanmean(per)), per

print("runner ready")

### 4 · Run the four configurations

In [ ]:
CONFIGS = [
    ("A  7B greedy",            "Qwen/Qwen2.5-7B-Instruct",  1),
    ("B  7B self-consistency",  "Qwen/Qwen2.5-7B-Instruct",  3),
    ("C  14B greedy",           "Qwen/Qwen2.5-14B-Instruct", 1),
    ("D  14B self-consistency", "Qwen/Qwen2.5-14B-Instruct", 3),
]

results = {}
T0 = time.time()
for name, model_name, n in CONFIGS:
    t0 = time.time()
    print(f"\n{'='*56}\n{name}\n{'='*56}", flush=True)
    cat = generate(model_name, SYS_CAT, reports, n_samples=n)
    num = generate(model_name, SYS_NUM, reports, n_samples=n)
    auc, per = score(cat, num)
    results[name] = (auc, per)
    fails = sum(1 for runs in cat + num for d in runs if not d)
    print(f"  macro AUC {auc:.4f}   (baseline 0.838)   parse failures {fails}   [{(time.time()-t0)/60:.1f} min]", flush=True)

print(f"\ntotal {(time.time()-T0)/60:.0f} min")

### 5 · Verdict

In [ ]:
BASE = 0.838
print(f"{'config':<26}{'macro AUC':>11}{'vs base':>10}")
for k, (a, _) in results.items():
    print(f"{k:<26}{a:>11.4f}{a-BASE:>+10.4f}")

best = max(results, key=lambda k: results[k][0])
gain = results[best][0] - BASE
print(f"\nbest: {best}  ({results[best][0]:.4f}, {gain:+.4f})")
print()
if gain < 0.010:
    print("Gain under +0.010. Not worth 10 GPU-hours of re-extraction plus")
    print("11 hours of retraining. Consider few-shot prompting instead, or")
    print("move on to MRNet pretraining.")
else:
    print("Worth re-extracting all 4,407 reports with this configuration.")

print("\nper-label, best config vs current extraction:")
CUR = {"ACL":0.881,"MCL":0.880,"Medial Meniscus":0.882,"Lateral Meniscus":0.761,
       "Medial OA":0.943,"Lateral OA":0.857,"PF OA":0.801,"Effusion":0.793,
       "Synovitis":0.729,"Baker's":0.891,"Contusion":0.799,"Fracture":0.844}
print(f"{'label':<18}{'current':>9}{'new':>9}{'delta':>9}")
for l, a in zip(LABELS, results[best][1]):
    c = CUR[l]
    flag = "  <--" if a - c > 0.03 else ""
    print(f"{l:<18}{c:>9.3f}{a:>9.3f}{a-c:>+9.3f}{flag}")